# New Best Model Analysis — Multi-seed CrossLead Deeper

**Source run:** `cv_results/repnet_crosslead_deeper_multiseed_2026-05-04_17-23-31/`

**Config:** `stage_filters=(48, 96, 192)`, `kernels=(7, 5, 3)`, `n_heads=4`,
`lr=2.465e-3`, `dropout=0.0546`, `weight_decay=1.67e-4`

**Multi-seed study:** 30 training seeds, 80/20 patient-grouped split (split_seed=42).  
**Best seed:** #64 (AUROC=0.8417)  |  **Ensemble (mean probs):** AUROC=0.7601

Sections:
1. Multi-seed AUROC distribution
2. Per-seed training curves
3. Train vs test distribution (best model)
4. ROC + Precision-Recall (best model)
5. Interactive threshold slider
6. Threshold sweep table + clinical-target thresholds
7. Feature-detection diagnostics: kernels, receptive field, cross-lead attention, saliency

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '../..')

import json
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, brier_score_loss,
)
from torch.utils.data import DataLoader, TensorDataset

from src.models.repnet_crosslead_deeper import RepNetCrossLeadDeeper
from src.data.dataset import load_seniordesign, split_holdout_grouped
from src.preprocessing.filters import BaselineWanderFilter, NotchFilter
from src.preprocessing.normalization import ZScoreNormalization

RUN_DIR    = Path('../../cv_results/repnet_crosslead_deeper_multiseed_2026-05-04_17-23-31')
DATA_DIR   = '../../data/seniordesign_upload'
SPLIT_SEED = 42

NET_PARAMS = dict(
    stage_filters=(48, 96, 192),
    kernels=(7, 5, 3),
    dropout=0.0546,
    n_heads=4,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Run:    {RUN_DIR.resolve()}')

## 1. Multi-seed AUROC distribution

In [ ]:
with open(RUN_DIR / 'results.json', encoding='utf-8') as f:
    results = json.load(f)

auroc_stats = results['auroc_stats']
per_seed    = results['per_seed']
ensemble    = results['ensemble']

aurocs = np.array([r['auroc'] for r in per_seed])
auprcs = np.array([r['auprc'] for r in per_seed])
briers = np.array([r['brier'] for r in per_seed])
seeds  = [r['seed'] for r in per_seed]

print(f"N seeds : {auroc_stats['n']}")
print(f"AUROC   : mean={auroc_stats['mean']:.4f}  std={auroc_stats['std']:.4f}  "
      f"sem={auroc_stats['sem']:.4f}")
print(f"          95% CI: [{auroc_stats['ci95_lo']:.4f}, {auroc_stats['ci95_hi']:.4f}]")
print(f"          min={auroc_stats['min']:.4f}  max={auroc_stats['max']:.4f}  "
      f"median={auroc_stats['median']:.4f}")
print(f"Best seed : #{results['best_seed']}  AUROC={results['best_auroc']:.4f}")
print(f"Ensemble  : AUROC={ensemble['auroc']:.4f}  AUPRC={ensemble['auprc']:.4f}  "
      f"Brier={ensemble['brier']:.4f}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=aurocs, nbinsx=min(20, max(5, len(aurocs) // 2)),
                           name='Seeds', marker_color='steelblue', opacity=0.8))
fig.add_vline(x=auroc_stats['mean'], line_dash='dash', line_color='red',
              annotation_text=f"mean={auroc_stats['mean']:.4f}",
              annotation_position='top right')
fig.add_vline(x=results['best_auroc'], line_dash='dot', line_color='green',
              annotation_text=f"best seed #{results['best_seed']}={results['best_auroc']:.4f}",
              annotation_position='top left')
fig.add_vline(x=ensemble['auroc'], line_dash='longdash', line_color='orange',
              annotation_text=f"ensemble={ensemble['auroc']:.4f}",
              annotation_position='bottom right')
fig.add_vrect(x0=auroc_stats['ci95_lo'], x1=auroc_stats['ci95_hi'],
              fillcolor='red', opacity=0.10, line_width=0,
              annotation_text='95% CI', annotation_position='top left')
fig.update_layout(
    title=f"Test AUROC distribution — {len(aurocs)} training seeds  "
          f"(mean={auroc_stats['mean']:.4f} ± {auroc_stats['std']:.4f})",
    xaxis_title='Test AUROC', yaxis_title='Count',
    template='plotly_white', width=800, height=440,
)
fig.show()

In [ ]:
# Per-seed bar chart sorted by AUROC
order = np.argsort(aurocs)[::-1]
colors = ['tomato' if seeds[i] == results['best_seed'] else 'steelblue' for i in order]
fig2 = go.Figure(go.Bar(
    x=[f"seed {seeds[i]}" for i in order],
    y=aurocs[order],
    marker_color=colors,
    text=[f"{aurocs[i]:.4f}" for i in order],
    textposition='outside',
))
fig2.add_hline(y=auroc_stats['mean'], line_dash='dash', line_color='red',
               annotation_text=f"mean={auroc_stats['mean']:.4f}")
fig2.update_layout(
    title='Per-seed test AUROC (red = best seed)',
    xaxis_title='Seed', yaxis_title='Test AUROC',
    yaxis_range=[max(0, aurocs.min() - 0.05), min(1, aurocs.max() + 0.05)],
    template='plotly_white', width=1100, height=460,
)
fig2.show()

## 2. Per-seed training curves

In [ ]:
with open(RUN_DIR / 'per_seed_history.json', encoding='utf-8') as f:
    histories = json.load(f)

with open(RUN_DIR / 'best_model_history.json', encoding='utf-8') as f:
    best_hist_data = json.load(f)

best_seed_key = str(results['best_seed'])

fig3 = go.Figure()
for sd_str, hist in histories.items():
    is_best = (sd_str == best_seed_key)
    ep = list(range(1, len(hist['val_auroc']) + 1))
    fig3.add_trace(go.Scatter(
        x=ep, y=hist['val_auroc'],
        name=f'seed {sd_str}' + ('  (best)' if is_best else ''),
        line=dict(width=3 if is_best else 1, color='red' if is_best else None),
        opacity=1.0 if is_best else 0.35,
        legendrank=0 if is_best else 1,
    ))
fig3.update_layout(
    title=f'Val AUROC per seed across {len(histories)} training runs (best=red)',
    xaxis_title='Epoch', yaxis_title='Val AUROC (early-stop set)',
    template='plotly_white', width=1100, height=480,
)
fig3.show()

In [ ]:
# Best-seed training curves
best_hist = best_hist_data['history']
ep = list(range(1, len(best_hist['train_loss']) + 1))
fig4 = make_subplots(rows=1, cols=2,
                     subplot_titles=('Train Loss', 'Val AUROC (early-stop set)'))
fig4.add_trace(go.Scatter(x=ep, y=best_hist['train_loss'], name='train_loss',
                          line=dict(color='steelblue')), row=1, col=1)
fig4.add_trace(go.Scatter(x=ep, y=best_hist['val_auroc'], name='val_auroc',
                          line=dict(color='tomato')), row=1, col=2)
best_ep = int(np.argmax(best_hist['val_auroc'])) + 1
fig4.add_vline(x=best_ep, line_dash='dot', line_color='green',
               annotation_text=f'best epoch {best_ep}', row=1, col=2)
fig4.update_layout(
    title=f"Best seed #{results['best_seed']} — test AUROC={results['best_auroc']:.4f}",
    template='plotly_white', width=1100, height=420,
)
fig4.show()

## 3. Load best model + data

In [ ]:
net = RepNetCrossLeadDeeper(**NET_PARAMS).to(device)
net.load_state_dict(torch.load(RUN_DIR / 'best_model.pt', map_location=device))
net.eval()
n_params = sum(p.numel() for p in net.parameters())
print(f'Loaded. Parameters: {n_params:,}')

In [ ]:
X, y, patient_ids = load_seniordesign(DATA_DIR, return_patient_ids=True)

flat_mask = (X.std(axis=2) < 1e-4).any(axis=1)
try:
    nan_mask = np.isnan(patient_ids.astype(float))
except (ValueError, TypeError):
    nan_mask = np.array([str(p).strip() in ('', 'nan', 'None') for p in patient_ids])
keep = ~flat_mask & ~nan_mask
X, y, patient_ids = X[keep], y[keep], patient_ids[keep]

X, _ = BaselineWanderFilter(cutoff=0.5, order=4, fs=250.0).transform(X)
X, _ = NotchFilter(freq=60.0, Q=30.0, fs=250.0).transform(X)
X, _ = ZScoreNormalization(per_lead=True).transform(X)

X_dev, X_test, y_dev, y_test, g_dev, g_test = split_holdout_grouped(
    X, y, patient_ids, test_size=0.20, seed=SPLIT_SEED,
)

print(f'Dev  : N={len(y_dev)}   PE={int(y_dev.sum())}   Normal={int((y_dev==0).sum())}   ({100*y_dev.mean():.1f}% pos)')
print(f'Test : N={len(y_test)}   PE={int(y_test.sum())}   Normal={int((y_test==0).sum())}   ({100*y_test.mean():.1f}% pos)')

In [ ]:
def infer(net, X, device, batch_size=64):
    Xt = torch.tensor(X, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt), batch_size=batch_size, num_workers=0)
    out = []
    with torch.no_grad():
        for (xb,) in dl:
            logits = net(xb.to(device))
            out.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

probs_dev  = infer(net, X_dev,  device)
probs_test = infer(net, X_test, device)

# Also load ensemble probs from the multiseed run
npz = np.load(RUN_DIR / 'all_probs.npz', allow_pickle=True)
probs_ensemble = npz['probs'].mean(axis=0)   # shape (N_test,)

auroc_dev  = roc_auc_score(y_dev,  probs_dev)
auroc_test = roc_auc_score(y_test, probs_test)
auroc_ens  = roc_auc_score(y_test, probs_ensemble)
auprc_dev  = average_precision_score(y_dev,  probs_dev)
auprc_test = average_precision_score(y_test, probs_test)
auprc_ens  = average_precision_score(y_test, probs_ensemble)

print(f'Dev  (best model) — AUROC: {auroc_dev:.4f}    AUPRC: {auprc_dev:.4f}')
print(f'Test (best model) — AUROC: {auroc_test:.4f}    AUPRC: {auprc_test:.4f}')
print(f'Test (ensemble)   — AUROC: {auroc_ens:.4f}    AUPRC: {auprc_ens:.4f}')
print(f'Generalization gap (best model): {auroc_dev - auroc_test:+.4f}')

## 4. Train vs test distribution (mirror histogram)

In [ ]:
def mirror_hist_traces(probs_, y_, nbins=40, show_legend=False):
    bins = np.linspace(0, 1, nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    width = bins[1] - bins[0]
    h_norm, _ = np.histogram(probs_[y_ == 0], bins=bins, density=True)
    h_pe,   _ = np.histogram(probs_[y_ == 1], bins=bins, density=True)
    return [
        go.Bar(x=centers, y=h_pe, name='PE',
               marker_color='tomato', width=width,
               showlegend=show_legend, legendgroup='PE'),
        go.Bar(x=centers, y=-h_norm, name='Normal',
               marker_color='steelblue', width=width,
               showlegend=show_legend, legendgroup='Normal'),
    ], h_norm.max(), h_pe.max()

fig5 = make_subplots(rows=1, cols=2,
    subplot_titles=(f'Dev  AUROC={auroc_dev:.3f}  N={len(y_dev)}',
                    f'Test AUROC={auroc_test:.3f}  N={len(y_test)}'))
ymax = 0.0
for col, (probs_, y_) in enumerate([(probs_dev, y_dev), (probs_test, y_test)], start=1):
    traces, hn, hp = mirror_hist_traces(probs_, y_, show_legend=(col == 1))
    ymax = max(ymax, hn, hp)
    for tr in traces:
        fig5.add_trace(tr, row=1, col=col)
    fig5.add_vline(x=0.5, line=dict(dash='dash', color='black'), row=1, col=col)
ymax *= 1.1
tickvals = np.linspace(-ymax, ymax, 7)
ticktext = [f'{abs(v):.1f}' for v in tickvals]
fig5.update_yaxes(tickvals=tickvals, ticktext=ticktext,
                  zeroline=True, zerolinecolor='black', zerolinewidth=1)
fig5.update_layout(template='plotly_white', barmode='overlay', bargap=0,
                   title='P(PE) — dev vs test  (best model, seed #64)  |  PE ↑, Normal ↓',
                   width=1100, height=460)
fig5.update_xaxes(title_text='P(PE)')
fig5.update_yaxes(title_text='Density (PE up / Normal down)', col=1)
fig5.show()

## 5. ROC + Precision-Recall (best model vs ensemble)

In [ ]:
fpr_dv, tpr_dv, _ = roc_curve(y_dev,  probs_dev)
fpr_te, tpr_te, _ = roc_curve(y_test, probs_test)
fpr_en, tpr_en, _ = roc_curve(y_test, probs_ensemble)
prec_dv, rec_dv, _ = precision_recall_curve(y_dev,  probs_dev)
prec_te, rec_te, _ = precision_recall_curve(y_test, probs_test)
prec_en, rec_en, _ = precision_recall_curve(y_test, probs_ensemble)

fig6 = make_subplots(rows=1, cols=2, subplot_titles=('ROC', 'Precision-Recall'))
fig6.add_trace(go.Scatter(x=fpr_dv, y=tpr_dv,
    name=f'Dev best model (AUC={auroc_dev:.3f})', line=dict(color='steelblue')), row=1, col=1)
fig6.add_trace(go.Scatter(x=fpr_te, y=tpr_te,
    name=f'Test best model (AUC={auroc_test:.3f})', line=dict(color='tomato')), row=1, col=1)
fig6.add_trace(go.Scatter(x=fpr_en, y=tpr_en,
    name=f'Test ensemble (AUC={auroc_ens:.3f})', line=dict(color='orange', dash='dot')), row=1, col=1)
fig6.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Random',
    line=dict(dash='dash', color='gray'), showlegend=False), row=1, col=1)
fig6.add_trace(go.Scatter(x=rec_dv, y=prec_dv,
    name=f'Dev best model (AP={auprc_dev:.3f})', line=dict(color='steelblue', dash='dot')), row=1, col=2)
fig6.add_trace(go.Scatter(x=rec_te, y=prec_te,
    name=f'Test best model (AP={auprc_test:.3f})', line=dict(color='tomato', dash='dot')), row=1, col=2)
fig6.add_trace(go.Scatter(x=rec_en, y=prec_en,
    name=f'Test ensemble (AP={auprc_ens:.3f})', line=dict(color='orange')), row=1, col=2)
fig6.update_xaxes(title_text='FPR', row=1, col=1)
fig6.update_yaxes(title_text='TPR', row=1, col=1)
fig6.update_xaxes(title_text='Recall', row=1, col=2)
fig6.update_yaxes(title_text='Precision', row=1, col=2)
fig6.update_layout(template='plotly_white', width=1100, height=480,
                   title='ROC & PR — best model vs ensemble (test)')
fig6.show()

## 6. Interactive threshold slider (test set, best model)

In [ ]:
def metrics_at(probs_, y_, tau):
    pred = (probs_ >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_, pred, labels=[0, 1]).ravel()
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    ppv  = tp / max(tp + fp, 1)
    npv  = tn / max(tn + fn, 1)
    f1   = 2 * ppv * sens / max(ppv + sens, 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn)
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                sens=sens, spec=spec, ppv=ppv, npv=npv, f1=f1, acc=acc)

thresholds   = np.round(np.arange(0.01, 1.00, 0.01), 2)
metric_table = [metrics_at(probs_test, y_test, t) for t in thresholds]

p_norm = probs_test[y_test == 0]
p_pe   = probs_test[y_test == 1]

nbins = 40
bins = np.linspace(0, 1, nbins + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
bar_w = bins[1] - bins[0]
h_norm, _ = np.histogram(p_norm, bins=bins, density=True)
h_pe,   _ = np.histogram(p_pe,   bins=bins, density=True)

fig7 = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38],
    specs=[[{'type': 'xy'}, {'type': 'heatmap'}]],
    subplot_titles=('Test P(PE) distribution  (PE ↑, Normal ↓)', 'Confusion matrix'))
fig7.add_trace(go.Bar(x=centers, y=h_pe,    name='PE',     marker_color='tomato',    width=bar_w), row=1, col=1)
fig7.add_trace(go.Bar(x=centers, y=-h_norm, name='Normal', marker_color='steelblue', width=bar_w), row=1, col=1)

init_tau = 0.50
init_idx = int(np.argmin(np.abs(thresholds - init_tau)))
init_m = metric_table[init_idx]
cm_z = [[init_m['tn'], init_m['fp']], [init_m['fn'], init_m['tp']]]
fig7.add_trace(go.Heatmap(z=cm_z, x=['Pred Normal', 'Pred PE'], y=['True Normal', 'True PE'],
    colorscale='Blues', showscale=False, text=cm_z, texttemplate='%{text}', textfont={'size': 18}), row=1, col=2)
fig7.add_shape(type='line', x0=init_tau, x1=init_tau, y0=0, y1=1,
               yref='paper', xref='x', line=dict(color='black', dash='dash', width=2))

def metric_text(t, m):
    return (f'<b>τ = {t:.2f}</b><br>'
            f'Sensitivity : {m["sens"]:.3f}<br>'
            f'Specificity : {m["spec"]:.3f}<br>'
            f'PPV         : {m["ppv"]:.3f}<br>'
            f'NPV         : {m["npv"]:.3f}<br>'
            f'F1          : {m["f1"]:.3f}<br>'
            f'Accuracy    : {m["acc"]:.3f}')
fig7.add_annotation(text=metric_text(init_tau, init_m),
    xref='paper', yref='paper', x=0.40, y=0.98, align='left', showarrow=False,
    bgcolor='rgba(255,255,255,0.85)', bordercolor='gray', borderwidth=1, borderpad=8,
    font=dict(family='monospace', size=12))

steps = []
for i, t in enumerate(thresholds):
    m = metric_table[i]
    cm_new = [[m['tn'], m['fp']], [m['fn'], m['tp']]]
    steps.append(dict(method='update', label=f'{t:.2f}',
        args=[{'z': [None, None, [cm_new]], 'text': [None, None, [cm_new]]},
              {'shapes': [dict(type='line', x0=float(t), x1=float(t), y0=0, y1=1,
                               yref='paper', xref='x', line=dict(color='black', dash='dash', width=2))],
               'annotations[2].text': metric_text(t, m)}]))

ymax = max(h_pe.max(), h_norm.max()) * 1.1
tickvals = np.linspace(-ymax, ymax, 7)
ticktext = [f'{abs(v):.1f}' for v in tickvals]
fig7.update_layout(
    sliders=[dict(active=init_idx, currentvalue={'prefix': 'τ = '}, pad={'t': 50}, steps=steps)],
    barmode='overlay', bargap=0, template='plotly_white', width=1200, height=560,
    title=f'Threshold tuner — best model seed #{results["best_seed"]} (test AUROC={auroc_test:.3f})')
fig7.update_xaxes(title_text='P(PE)', row=1, col=1)
fig7.update_yaxes(title_text='Density (PE up / Normal down)', tickvals=tickvals, ticktext=ticktext,
                  zeroline=True, zerolinecolor='black', zerolinewidth=1, row=1, col=1)
fig7.show()

## 7. Threshold sweep + clinical-target thresholds

In [ ]:
key_taus = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
rows = []
for t in key_taus:
    m = metrics_at(probs_test, y_test, t)
    rows.append({'τ': t, 'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'], 'TN': m['tn'],
                 'Sens': round(m['sens'], 3), 'Spec': round(m['spec'], 3),
                 'PPV': round(m['ppv'], 3), 'NPV': round(m['npv'], 3),
                 'F1': round(m['f1'], 3), 'Acc': round(m['acc'], 3)})
pd.DataFrame(rows).set_index('τ')

In [ ]:
def threshold_for_target(probs_, y_, target_sens=0.85):
    fpr, tpr, thr = roc_curve(y_, probs_)
    valid = tpr >= target_sens
    if not valid.any():
        return None
    i = int(np.argmax(thr[valid]))
    return float(thr[valid][i])

for target in [0.95, 0.90, 0.85, 0.80]:
    t = threshold_for_target(probs_test, y_test, target)
    if t is None:
        print(f'target sens {target}: not achievable')
        continue
    m = metrics_at(probs_test, y_test, t)
    print(f'sens >= {target} → τ={t:.3f}  sens={m["sens"]:.3f}  spec={m["spec"]:.3f}  '
          f'PPV={m["ppv"]:.3f}  F1={m["f1"]:.3f}  (TP={m["tp"]}, FP={m["fp"]}, FN={m["fn"]}, TN={m["tn"]})')

In [ ]:
from sklearn.metrics import classification_report

fpr_te2, tpr_te2, thr_te2 = roc_curve(y_test, probs_test)
tau_youden  = float(thr_te2[np.argmax(tpr_te2 - fpr_te2)])
valid       = tpr_te2 >= 0.85
tau_sens85  = float(thr_te2[valid][np.argmax(thr_te2[valid])]) if valid.any() else 0.0

operating_points = [
    ('tau = 0.50  (default)',                     0.50),
    (f'tau = {tau_youden:.3f}  (Youden J)',        tau_youden),
    (f'tau = {tau_sens85:.3f}  (sens >= 0.85)',    tau_sens85),
]

for name, tau in operating_points:
    pred = (probs_test >= tau).astype(int)
    print(f'\n=== {name} ===')
    print(classification_report(y_test, pred, target_names=['Normal', 'PE'],
                                digits=3, zero_division=0))

---

# 8. Feature-detection diagnostics

## 8a. First-stage conv kernels

In [ ]:
k1 = net.stages[0]['conv'].conv1.weight.detach().cpu().squeeze(1).numpy()  # (48, 7)
order = np.argsort(-np.linalg.norm(k1, axis=1))
k1_sorted = k1[order]

fig_k = go.Figure(data=go.Heatmap(z=k1_sorted, colorscale='RdBu_r', zmid=0,
                                   colorbar=dict(title='weight')))
fig_k.update_layout(
    title=f'Stage 1 conv1 kernels ({k1.shape[0]} filters × {k1.shape[1]} taps), sorted by L2 norm',
    xaxis_title='kernel tap (4 ms each)', yaxis_title='filter idx (sorted)',
    template='plotly_white', width=620, height=620)
fig_k.show()

## 8b. Receptive field

In [ ]:
def rf_table(kernels, fs=250.0):
    rf, eff_stride = 1, 1
    rows = []
    for stage_idx, k in enumerate(kernels, start=1):
        rf += (k - 1) * eff_stride
        rows.append({'stage': stage_idx, 'layer': 'conv1', 'k': k, 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
        rf += (k - 1) * eff_stride
        rows.append({'stage': stage_idx, 'layer': 'conv2', 'k': k, 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
        eff_stride *= 2
        rows.append({'stage': stage_idx, 'layer': 'pool', 'k': '-', 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
    return pd.DataFrame(rows)

rf_df = rf_table(NET_PARAMS['kernels'], fs=250.0)
print(f"Total RF: {rf_df['RF (samples)'].iloc[-1]} samples = {rf_df['RF (ms)'].iloc[-1]} ms @ 250 Hz")
rf_df

## 8c. Cross-lead attention (per-stage, per-class)

In [ ]:
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
attn_storage = {f'stage{i+1}': [] for i in range(len(net.stages))}

def make_attn_hook(name):
    def hook(module, inputs, output):
        if isinstance(output, tuple) and len(output) >= 2 and output[1] is not None:
            attn_storage[name].append(output[1].detach().cpu().numpy())
    return hook

handles = []
for i, stage in enumerate(net.stages):
    handles.append(stage['attn'].attn.register_forward_hook(make_attn_hook(f'stage{i+1}')))

with torch.no_grad():
    Xt = torch.tensor(X_test, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt), batch_size=64)
    for (xb,) in dl:
        net(xb.to(device))

for h in handles:
    h.remove()

attn_by_stage = {k: np.concatenate(v, axis=0) for k, v in attn_storage.items()}
n_stages = len(attn_by_stage)

fig_a = make_subplots(rows=2, cols=n_stages,
    subplot_titles=[f'Stage {i+1} — Normal' for i in range(n_stages)] +
                   [f'Stage {i+1} — PE' for i in range(n_stages)],
    horizontal_spacing=0.08, vertical_spacing=0.12)
for col, k in enumerate([f'stage{i+1}' for i in range(n_stages)], start=1):
    A = attn_by_stage[k]
    A_norm = A[y_test == 0].mean(axis=0)
    A_pe   = A[y_test == 1].mean(axis=0)
    fig_a.add_trace(go.Heatmap(z=A_norm, x=LEADS, y=LEADS,
                               colorscale='Blues', showscale=(col == n_stages),
                               colorbar=dict(x=1.02, len=0.4, y=0.78)),
                    row=1, col=col)
    fig_a.add_trace(go.Heatmap(z=A_pe, x=LEADS, y=LEADS,
                               colorscale='Reds', showscale=(col == n_stages),
                               colorbar=dict(x=1.02, len=0.4, y=0.22)),
                    row=2, col=col)
fig_a.update_layout(template='plotly_white', width=1200, height=700,
                    title='Mean cross-lead attention per stage (rows=query, cols=key)')
fig_a.show()

## 8d. Saliency heatmap (Integrated Gradients)

In [ ]:
from scipy.ndimage import gaussian_filter1d

def integrated_gradients(net, x_np, target_class=1, n_steps=32, baseline=None, smooth_sigma=5.0):
    x    = torch.tensor(x_np, dtype=torch.float32, device=device)
    base = (torch.zeros_like(x) if baseline is None
            else torch.tensor(baseline, dtype=torch.float32, device=device))
    alphas = torch.linspace(0.5/n_steps, 1.0-0.5/n_steps, n_steps, device=device).view(-1, 1, 1)
    interp = base.unsqueeze(0) + alphas * (x - base).unsqueeze(0)
    interp.requires_grad_(True)
    net.zero_grad()
    logits = net(interp)
    grads  = torch.autograd.grad(logits[:, target_class].sum(), interp)[0]
    avg_grad = grads.mean(dim=0)
    attribution = ((x - base) * avg_grad).cpu().numpy()
    if smooth_sigma > 0:
        attribution = gaussian_filter1d(attribution, sigma=smooth_sigma, axis=1)
    return attribution

def pick_8_cases(probs, y):
    pred = (probs >= 0.5).astype(int)
    tp = np.where((y == 1) & (pred == 1))[0]
    tn = np.where((y == 0) & (pred == 0))[0]
    fp = np.where((y == 0) & (pred == 1))[0]
    fn = np.where((y == 1) & (pred == 0))[0]
    out = {}
    if len(tp): out['confident_TP']  = tp[np.argmax(probs[tp])]
    if len(tn): out['confident_TN']  = tn[np.argmin(probs[tn])]
    if len(fp): out['confident_FP']  = fp[np.argmax(probs[fp])]
    if len(fn): out['confident_FN']  = fn[np.argmin(probs[fn])]
    if len(tp): out['borderline_TP'] = tp[np.argmin(np.abs(probs[tp] - 0.5))]
    if len(tn): out['borderline_TN'] = tn[np.argmin(np.abs(probs[tn] - 0.5))]
    if len(fp): out['borderline_FP'] = fp[np.argmin(np.abs(probs[fp] - 0.5))]
    if len(fn): out['borderline_FN'] = fn[np.argmin(np.abs(probs[fn] - 0.5))]
    return out

cases   = pick_8_cases(probs_test, y_test)
ordered = list(cases.items())
print('Selected examples:')
for label, idx in ordered:
    print(f'  {label:<18}  idx={idx:>4}  true={int(y_test[idx])}  P(PE)={probs_test[idx]:.3f}')

attributions     = {idx: integrated_gradients(net, X_test[idx]) for idx in cases.values()}

In [ ]:
def plot_heatmap_for_case(label, idx, top_k=4, fs=250.0):
    attr = attributions[idx]
    ecg  = X_test[idx]
    t    = np.arange(ecg.shape[1]) / fs
    lead_strength = np.abs(attr).sum(axis=1)
    top_leads = list(np.argsort(-lead_strength)[:top_k])
    cmax = float(np.abs(attr[top_leads]).max()) or 1.0

    fig = make_subplots(rows=top_k, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=[f'Lead {LEADS[L]}' for L in top_leads])
    for r, L in enumerate(top_leads, start=1):
        ymin, ymax = float(ecg[L].min()), float(ecg[L].max())
        ypad = (ymax - ymin) * 0.15
        fig.add_trace(go.Heatmap(
            x=t, y=[ymin - ypad, ymax + ypad],
            z=[attr[L], attr[L]],
            colorscale='RdBu_r', zmid=0, zmin=-cmax, zmax=cmax,
            showscale=(r == 1),
            colorbar=dict(title='IG attr<br>(signed)', x=1.02, len=0.9) if r == 1 else None,
            opacity=0.55,
        ), row=r, col=1)
        fig.add_trace(go.Scatter(
            x=t, y=ecg[L], mode='lines',
            line=dict(color='black', width=1.2), showlegend=False,
        ), row=r, col=1)
    pred       = int(probs_test[idx] >= 0.5)
    true_label = 'PE' if y_test[idx] == 1 else 'Normal'
    pred_label = 'PE' if pred == 1 else 'Normal'
    correct    = pred == int(y_test[idx])
    fig.update_layout(
        template='plotly_white', width=1200, height=180 * top_k + 120,
        title=(f'<b>{label.replace("_", " ")}</b>  |  idx {idx}  |  '
               f'true={true_label}  pred={pred_label}  P(PE)={probs_test[idx]:.3f}  |  '
               f'{"correct" if correct else "WRONG"}<br>'
               f'<sub>Red = pushes toward PE, Blue = toward Normal  |  ±{cmax:.2e}</sub>'),
    )
    fig.update_xaxes(title_text='time (s)', row=top_k, col=1)
    return fig

for label, idx in ordered:
    plot_heatmap_for_case(label, idx, top_k=4).show()

## 8e. Grad-CAM

In [ ]:
import torch.nn.functional as F

def gradcam_per_lead(net, x_np, target_class=1, smooth_sigma=2.0):
    x = torch.tensor(x_np, dtype=torch.float32, device=device).unsqueeze(0)
    activations = {}
    def fwd_hook(_m, _i, o):
        activations['feat'] = o
        o.retain_grad()
    h = net.stages[-1]['conv'].register_forward_hook(fwd_hook)
    net.zero_grad()
    logits = net(x)
    logits[0, target_class].backward()
    h.remove()
    feat    = activations['feat']
    weights = feat.grad.mean(dim=(0, 3))
    cam     = (weights.unsqueeze(-1) * feat[0]).sum(dim=1)
    T = x_np.shape[1]
    cam_full = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(12, T),
                             mode='bilinear', align_corners=False)
    cam_full = cam_full.squeeze().detach().cpu().numpy()
    if smooth_sigma > 0:
        cam_full = gaussian_filter1d(cam_full, sigma=smooth_sigma, axis=1)
    return cam_full

gradcams = {idx: gradcam_per_lead(net, X_test[idx]) for idx in cases.values()}

def plot_gradcam_for_case(label, idx, top_k=4, fs=250.0):
    cam = gradcams[idx]
    ecg = X_test[idx]
    t   = np.arange(ecg.shape[1]) / fs
    top_leads = list(np.argsort(-np.abs(cam).sum(axis=1))[:top_k])
    cmax = float(np.abs(cam[top_leads]).max()) or 1.0
    fig = make_subplots(rows=top_k, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=[f'Lead {LEADS[L]}' for L in top_leads])
    for r, L in enumerate(top_leads, start=1):
        ymin, ymax = float(ecg[L].min()), float(ecg[L].max())
        ypad = (ymax - ymin) * 0.15
        fig.add_trace(go.Heatmap(
            x=t, y=[ymin - ypad, ymax + ypad],
            z=[cam[L], cam[L]],
            colorscale='RdBu_r', zmid=0, zmin=-cmax, zmax=cmax,
            showscale=(r == 1),
            colorbar=dict(title='Grad-CAM<br>(signed)', x=1.02, len=0.9) if r == 1 else None,
            opacity=0.55,
        ), row=r, col=1)
        fig.add_trace(go.Scatter(
            x=t, y=ecg[L], mode='lines',
            line=dict(color='black', width=1.2), showlegend=False,
        ), row=r, col=1)
    pred       = int(probs_test[idx] >= 0.5)
    true_label = 'PE' if y_test[idx] == 1 else 'Normal'
    pred_label = 'PE' if pred == 1 else 'Normal'
    correct    = pred == int(y_test[idx])
    fig.update_layout(
        template='plotly_white', width=1200, height=180 * top_k + 120,
        title=(f'<b>Grad-CAM — {label.replace("_", " ")}</b>  |  idx {idx}  |  '
               f'true={true_label}  pred={pred_label}  P(PE)={probs_test[idx]:.3f}  |  '
               f'{"correct" if correct else "WRONG"}<br>'
               f'<sub>Red = pushes toward PE, Blue = toward Normal  |  ±{cmax:.2e}</sub>'),
    )
    fig.update_xaxes(title_text='time (s)', row=top_k, col=1)
    return fig

for label, idx in ordered:
    plot_gradcam_for_case(label, idx, top_k=4).show()

## 8f. Layer-by-layer activation tracking (confident TP)

In [ ]:
TRACK_LEAD = 1
TRACK_IDX  = cases['confident_TP']
print(f'Tracking idx={TRACK_IDX}, lead={LEADS[TRACK_LEAD]}, P(PE)={probs_test[TRACK_IDX]:.3f}')

acts = {}
def cap(name):
    def hook(_m, _i, output):
        if isinstance(output, tuple):
            output = output[0]
        acts[name] = output.detach().cpu()
    return hook

handles = [
    net.stages[0]['conv'].register_forward_hook(cap('s1_conv')),
    net.stages[0]['attn'].register_forward_hook(cap('s1_attn')),
    net.stages[1]['conv'].register_forward_hook(cap('s2_conv')),
    net.stages[1]['attn'].register_forward_hook(cap('s2_attn')),
    net.stages[2]['conv'].register_forward_hook(cap('s3_conv')),
    net.stages[2]['attn'].register_forward_hook(cap('s3_attn')),
    net.fuse.register_forward_hook(cap('fuse')),
    net.gap.register_forward_hook(cap('gap')),
    net.fc.register_forward_hook(cap('fc')),
]

with torch.no_grad():
    x_in = torch.tensor(X_test[TRACK_IDX:TRACK_IDX+1], dtype=torch.float32, device=device)
    logits_tr = net(x_in)
    probs_tr  = torch.softmax(logits_tr, dim=1)

for h in handles:
    h.remove()

print(f'Logits  = {logits_tr.cpu().numpy().ravel()}')
print(f'Softmax = {probs_tr.cpu().numpy().ravel()}  -> P(PE) = {probs_tr[0, 1].item():.4f}')
print('\nActivation shapes:')
for name, t in acts.items():
    print(f'  {name:<10}  {tuple(t.shape)}')

In [ ]:
ecg_lead = X_test[TRACK_IDX, TRACK_LEAD, :]
t_input  = np.arange(2500) / 250.0

stage_keys = [
    ('Stage 1: PerLeadConv (48 ch)',   's1_conv'),
    ('Stage 1: CrossLeadAttn output',  's1_attn'),
    ('Stage 2: PerLeadConv (96 ch)',   's2_conv'),
    ('Stage 2: CrossLeadAttn output',  's2_attn'),
    ('Stage 3: PerLeadConv (192 ch)',  's3_conv'),
    ('Stage 3: CrossLeadAttn output',  's3_attn'),
]

n_rows = 1 + len(stage_keys)
fig_l = make_subplots(
    rows=n_rows, cols=1, shared_xaxes=False, vertical_spacing=0.025,
    row_heights=[0.10] + [0.15] * len(stage_keys),
    subplot_titles=[f'Input ECG  Lead {LEADS[TRACK_LEAD]}'] +
                   [f'{name}  --  {tuple(acts[k].shape[2:])}' for name, k in stage_keys],
)
fig_l.add_trace(go.Scatter(x=t_input, y=ecg_lead, mode='lines',
                           line=dict(color='black', width=1.2), showlegend=False), row=1, col=1)

for r, (name, key) in enumerate(stage_keys, start=2):
    a = acts[key][0, TRACK_LEAD].numpy()
    C, T_stage = a.shape
    t_stage = np.linspace(0, 10.0, T_stage)
    cmax = float(np.abs(a).max()) or 1.0
    fig_l.add_trace(go.Heatmap(
        x=t_stage, y=np.arange(C), z=a,
        colorscale='RdBu_r', zmid=0, zmin=-cmax, zmax=cmax,
        showscale=False,
    ), row=r, col=1)

fig_l.update_xaxes(range=[0, 10.0])
fig_l.update_yaxes(title_text='mV (z-scored)', row=1, col=1)
for r in range(2, n_rows + 1):
    fig_l.update_yaxes(title_text='channel', row=r, col=1)
fig_l.update_xaxes(title_text='time (s)', row=n_rows, col=1)
fig_l.update_layout(
    template='plotly_white', width=1200, height=180 * n_rows + 100,
    title=f'<b>Layer-by-layer activations</b>  |  idx {TRACK_IDX} (confident TP)  |  '
          f'P(PE) = {probs_test[TRACK_IDX]:.3f}',
)
fig_l.show()

# Fusion + head
fuse_out = acts['fuse'][0].numpy()
gap_out  = acts['gap'][0, :, 0].numpy()
fc_out   = acts['fc'][0].numpy()
sm_out   = probs_tr[0].cpu().numpy()
C_fuse, T_fuse = fuse_out.shape

fig_h = make_subplots(
    rows=3, cols=1, row_heights=[0.55, 0.25, 0.20], vertical_spacing=0.10,
    subplot_titles=(
        f'Fusion output  ({C_fuse} ch × {T_fuse} time)',
        f'GAP output  ({len(gap_out)}-dim vector)',
        f'Logits → Softmax → P(PE) = {sm_out[1]:.4f}',
    ),
)
cmax_f = float(np.abs(fuse_out).max()) or 1.0
fig_h.add_trace(go.Heatmap(x=np.linspace(0, 10, T_fuse), y=np.arange(C_fuse), z=fuse_out,
    colorscale='RdBu_r', zmid=0, zmin=-cmax_f, zmax=cmax_f, showscale=True,
    colorbar=dict(title='fusion act', x=1.02, len=0.5, y=0.78)), row=1, col=1)
fig_h.add_trace(go.Bar(x=np.arange(len(gap_out)), y=gap_out,
    marker_color=['tomato' if v > 0 else 'steelblue' for v in gap_out], showlegend=False), row=2, col=1)
fig_h.add_hline(y=0, line=dict(color='black', width=1), row=2, col=1)
fig_h.add_trace(go.Bar(
    x=['Normal logit', 'PE logit', 'Normal P', 'PE P'],
    y=[fc_out[0], fc_out[1], sm_out[0], sm_out[1]],
    marker_color=['steelblue', 'tomato', 'lightblue', 'lightcoral'],
    text=[f'{fc_out[0]:.2f}', f'{fc_out[1]:.2f}', f'{sm_out[0]:.3f}', f'{sm_out[1]:.3f}'],
    textposition='outside', showlegend=False,
), row=3, col=1)
fig_h.update_layout(template='plotly_white', width=1200, height=900,
    title=f'<b>Fusion + classifier head</b>  |  idx {TRACK_IDX}  |  P(PE) = {probs_test[TRACK_IDX]:.4f}')
fig_h.show()